In [ ]:
import pandas as pd
import os

def generate_summary():
    # A fájlok nevei és a hozzájuk tartozó modell nevek
    files = {
        "Florence-2": "/kaggle/input/datasets/gonoszgonosz/out-of-box-results-1/csv/florence2_comprehensive_metrics.csv",
        "Grounded-SAM 2": "/kaggle/input/datasets/gonoszgonosz/out-of-box-results-1/csv/grounded_sam2_metrics.csv",
        "SAM 2 (Oracle)": "/kaggle/input/datasets/gonoszgonosz/out-of-box-results-1/csv/sam2_comprehensive_metrics.csv"
    }

    summary_data = []

    for model_name, file_name in files.items():
        if os.path.exists(file_name):
            df = pd.read_csv(file_name)
            
            # A Grounded-SAM 2 esetében az augmentált frame-eknél lehetnek üres (NaN) 
            # vagy nulla értékek, ha a detektálás teljesen elbukott. 
            # A .mean() alapértelmezetten kihagyja a NaN értékeket, de a nullákat belevéve 
            # reális képet kapunk a modell robusztusságáról.
            
            summary = {
                "Modell": model_name,
                "mIoU": df["mIoU"].mean(),
                "Dice Score": df["Dice"].mean(),
                "Boundary IoU": df["Boundary_IoU"].mean(),
                "Hausdorff Távolság": df["Hausdorff_Distance"].mean()
            }
            summary_data.append(summary)
        else:
            print(f"Hiba: A {file_name} fájl nem található a mappában!")

    if not summary_data:
        return

    # Összesítő DataFrame létrehozása
    summary_df = pd.DataFrame(summary_data)

    # Kerekítés 4 tizedesjegyre
    summary_df = summary_df.round(4)

    print("\n=== ZERO-SHOT MODELLEK ÖSSZEHASONLÍTÁSA ===")
    print(summary_df.to_string(index=False))
    
    # Készítünk egy LaTeX kimenetet is, amit egy az egyben be lehet másolni a szakdolgozatba
    print("\n=== LATEX TÁBLÁZAT A SZAKDOLGOZATBA ===")
    latex_table = summary_df.to_latex(index=False, caption="A vizsgált zero-shot modellek kvantitatív összehasonlítása az etológiai adathalmazon", label="tab:zero_shot_results")
    print(latex_table)

if __name__ == "__main__":
    generate_summary()